# Check Prompt Evaluation Workflow

This notebook automates the process of evaluating and improving model-based check prompts using Okareo's meta-evaluation methodology.

## Overview

This workflow:
1. Extracts test run data from existing evaluations
2. Transforms data into scenario format for evaluation
3. Registers check-as-target models with different prompt versions
4. Runs evaluations and compares results
5. Enables iterative prompt improvement

## Goals

After using this notebook, you will be able to:
- Extract and analyze test run data
- Create scenario sets from test run data
- Register check-as-target models with prompt templates
- Run evaluations and compare prompt versions
- Iteratively improve check prompts based on evaluation results

## Step 1: Setup & Connection

First, let's set up the Okareo SDK connection and verify everything is working.

In [ ]:
# Install required packages if needed (can be skipped if already installed)
%pip install --upgrade okareo ipywidgets

In [ ]:
# Configuration: Set these variables for your evaluation project
import os

# API Keys (from environment variables)
OKAREO_API_KEY = os.environ.get("LOCAL_OKAREO_API_KEY", "<YOUR_OKAREO_API_KEY>")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "<OPENAI_API_KEY>")

# Project and Evaluation Configuration
project_id = "<YOUR_PROJECT_ID>"  # Your Okareo project ID
test_run_id = "<YOUR_TEST_RUN_ID>"  # Source test run ID to extract data from
check_name = "<YOUR_CHECK_NAME>"  # Name of the check being evaluated

# API Configuration
BASE_PATH = "http://localhost:8000"  # Base path for API calls (use localhost:8000 for local, or api.okareo.com for production)

In [ ]:
from okareo import Okareo

okareo = Okareo(OKAREO_API_KEY, base_path=BASE_PATH)
print("✅ Successfully connected to Okareo!")

## Step 2: Extract Test Run Data

Extract test run data points from an existing evaluation. This is the source data we'll use to create scenarios for evaluating our check.

In [ ]:
# Extract test run data points
# Set full_data_point=True to get complete scenario and model information
from okareo_api_client.models.find_test_data_point_payload import FindTestDataPointPayload

payload = FindTestDataPointPayload(
    test_run_id=test_run_id,
    full_data_point=True
)
test_data_points = okareo.find_test_data_points(payload)

print(f"✅ Extracted {len(test_data_points)} data points from test run {test_run_id}")
print(f"\nFirst data point structure:")
if test_data_points:
    import json
    first_point = test_data_points[0]
    # Convert Pydantic model to dict for display
    first_point_dict = first_point.to_dict()
    print(f"Keys: {list(first_point_dict.keys())}")
    print(f"\nSample data point (first 10 fields):")
    print(json.dumps(dict(list(first_point_dict.items())[:10]), indent=2, default=str))


In [ ]:
# Display data points in a readable table format
import pandas as pd

# Extract key information from data points for display
display_data = []
for i, dp in enumerate(test_data_points[:20]):  # Show first 20
    # Convert Pydantic model to dict for easier access
    dp_dict = dp.to_dict()
    checks = dp_dict.get('checks', {}) if isinstance(dp_dict.get('checks'), dict) else {}
    display_data.append({
        'Index': i,
        'ID': str(dp_dict.get('id', 'N/A'))[:8] + '...',
        'Scenario Input': str(dp_dict.get('scenario_input', 'N/A'))[:50],
        'Has Check Result': check_name in checks,
        'Check Result': checks.get(check_name, 'N/A'),
    })

df = pd.DataFrame(display_data)
print(f"Showing first {min(20, len(test_data_points))} of {len(test_data_points)} data points:")
print(df.to_string(index=False))

if len(test_data_points) > 20:
    print(f"\n... and {len(test_data_points) - 20} more data points")


## Step 3: Transform Data to Scenario Format

Transform the extracted test run data points into scenario format that can be used for evaluating the check. We'll include all 5 core template variables in every scenario:
- `generation` / `model_output` - model's actual output
- `input` / `scenario_input` - original scenario input
- `result` / `scenario_result` - expected result/ground truth
- `tool_calls` - tool calls made by the model (if any)
- `tools` - available tools (if any)


In [ ]:
# Helper functions for data transformation
import json

def parse_json_or_string(value):
    """Parse JSON string if possible, otherwise return as-is."""
    if value is None:
        return None
    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return value
    return value

def format_tools_json(tools):
    """Format tools as readable JSON with indentation."""
    if tools is None:
        return "[]"
    try:
        parsed = parse_json_or_string(tools)
        return json.dumps(parsed, indent=4)
    except Exception:
        return str(tools) if tools else "[]"


In [ ]:
# Transform test run data points to scenario format
scenario_data = []
skipped = 0

for dp in test_data_points:
    dp_dict = dp.to_dict()
    
    # Extract core fields (all 5 template variables)
    scenario_result = parse_json_or_string(dp_dict.get('scenario_result'))
    scenario_input = parse_json_or_string(dp_dict.get('scenario_input'))
    model_result = parse_json_or_string(dp_dict.get('model_result'))
    
    # Extract metadata fields
    metadata = dp_dict.get('model_metadata', {}) or {}
    tool_calls = parse_json_or_string(metadata.get('tool_calls'))
    tools = metadata.get('tools')
    
    # Extract check result for this check (ground truth)
    checks = dp_dict.get('checks', {}) or {}
    check_result = checks.get(check_name)
    
    # Skip if required fields are missing
    if scenario_result is None or model_result is None:
        skipped += 1
        continue
    
    # Create scenario row with nested structure (all 5 template variables under "input")
    scenario_row = {
        "input": {
            "generation": model_result,        # model_output
            "result": scenario_result,         # scenario_result (expected result)
            "input": scenario_input,           # scenario_input
            "tool_calls": tool_calls or "",    # tool_calls (empty string if None)
            "tools": format_tools_json(tools)  # tools (formatted JSON string)
        },
        "result": check_result                 # check result (ground truth for evaluation)
    }
    
    scenario_data.append(scenario_row)

print(f"✅ Transformed {len(scenario_data)} data points to scenario format")
if skipped > 0:
    print(f"⚠ Skipped {skipped} data points (missing required fields)")


In [ ]:
# Display transformed scenario data for verification
print(f"Sample scenario data (first 3 rows):\n")

for i, row in enumerate(scenario_data[:3]):
    print(f"Row {i+1}:")
    input_data = row['input']
    print(f"  input.generation: {str(input_data['generation'])[:100]}..." if len(str(input_data['generation'])) > 100 else f"  input.generation: {input_data['generation']}")
    print(f"  input.input: {str(input_data['input'])[:100]}..." if len(str(input_data['input'])) > 100 else f"  input.input: {input_data['input']}")
    print(f"  input.result: {str(input_data['result'])[:100]}..." if len(str(input_data['result'])) > 100 else f"  input.result: {input_data['result']}")
    print(f"  input.tool_calls: {input_data['tool_calls']}")
    print(f"  input.tools: {input_data['tools'][:100]}..." if len(input_data['tools']) > 100 else f"  input.tools: {input_data['tools']}")
    print(f"  result (check result): {row['result']}")
    print()


## Step 4: Review & Correct Ground Truth

Review and correct the check results (ground truth) for each scenario. Use the interactive widgets below to:
- Filter scenarios by current result value
- Change individual scenario results using the dropdowns
- Save your changes back to the scenario data


In [ ]:
# Interactive Ground Truth Review Interface
import ipywidgets as widgets
from IPython.display import display, clear_output
import copy
import json

# Configure which columns to display
# Options: "input", "result", "generation", "tools", "tool_calls"
columns_to_display = ["result", "generation"]  # Change this list to customize visible columns

# Store original data for reset functionality
original_scenario_data = copy.deepcopy(scenario_data)

# Helper function to format column value for display
def format_column_value(value):
    """Format a column value for display in HTML widget."""
    if value is None:
        return "<em>None</em>"
    if isinstance(value, (dict, list)):
        # Use CSS to constrain width and enable scrolling
        json_str = json.dumps(value, indent=2)
        return f"<pre style='max-width: 400px; overflow-x: auto; white-space: pre-wrap; word-wrap: break-word; margin: 0;'>{json_str}</pre>"
    # For string values, also constrain width and enable wrapping
    return f"<pre style='max-width: 400px; overflow-x: auto; white-space: pre-wrap; word-wrap: break-word; margin: 0;'>{str(value)}</pre>"

# Create filter dropdown (True/False only, no None)
filter_dropdown = widgets.Dropdown(
    options=[
        ('All', 'all'),
        ('True', True),
        ('False', False)
    ],
    value='all',
    description='Filter by result:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Create dropdown widgets and column displays for each row
row_widgets = []
row_containers = []

for i, row in enumerate(scenario_data):
    current_result = row['result']
    # Convert None to False if needed (since we only allow True/False now)
    if current_result is None:
        current_result = False
    
    # Create dropdown for this row (True/False only)
    dropdown = widgets.Dropdown(
        options=[True, False],
        value=current_result if current_result in [True, False] else False,
        description=f'Row {i}:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='200px')
    )
    
    # Create column display widgets
    column_widgets = []
    for col_name in columns_to_display:
        # Get the value from the nested structure
        if col_name == "result":
            col_value = row['input'].get('result', None)
        elif col_name == "generation":
            col_value = row['input'].get('generation', None)
        elif col_name == "input":
            col_value = row['input'].get('input', None)
        elif col_name == "tools":
            col_value = row['input'].get('tools', None)
        elif col_name == "tool_calls":
            col_value = row['input'].get('tool_calls', None)
        else:
            col_value = None
        
        # Format and display the column value (full content, no truncation)
        formatted_value = format_column_value(col_value)
        col_label = widgets.HTML(
            value=f"<div style='width: 400px; max-width: 400px; overflow-x: auto; padding: 10px; box-sizing: border-box;'><strong>{col_name}:</strong><br/>{formatted_value}</div>",
            layout=widgets.Layout(width='400px', padding='10px')
        )
        column_widgets.append(col_label)
    
    # Container for this row: columns first, then dropdown at the end
    row_items = column_widgets + [dropdown]
    row_container = widgets.HBox(row_items, layout=widgets.Layout(border='1px solid gray', padding='5px', margin='5px'))
    
    row_widgets.append(dropdown)
    row_containers.append(row_container)

# Function to filter and display rows
def update_display(filter_value):
    """Update the displayed rows based on filter."""
    filtered_containers = []
    
    for i, row in enumerate(scenario_data):
        current_result = row['result']
        if current_result is None:
            current_result = False
        if filter_value == 'all' or current_result == filter_value:
            filtered_containers.append(row_containers[i])
    
    return widgets.VBox(filtered_containers)

# Create output area for filtered rows
output_area = widgets.Output()

def on_filter_change(change):
    """Handle filter dropdown change."""
    with output_area:
        clear_output(wait=True)
        filtered = update_display(change['new'])
        display(filtered)

filter_dropdown.observe(on_filter_change, names='value')

# Create save button
save_button = widgets.Button(
    description='Save Changes',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='150px', margin='10px 0px')
)

# Create reset button
reset_button = widgets.Button(
    description='Reset to Original',
    button_style='warning',
    icon='undo',
    layout=widgets.Layout(width='150px', margin='10px 0px')
)

# Summary output area
summary_output = widgets.Output()

def on_save_clicked(b):
    """Handle save button click."""
    changes = []
    for i, dropdown in enumerate(row_widgets):
        old_value = original_scenario_data[i]['result']
        new_value = dropdown.value
        if old_value != new_value:
            scenario_data[i]['result'] = new_value
            changes.append({
                'row': i,
                'old': old_value,
                'new': new_value
            })
    
    with summary_output:
        clear_output(wait=True)
        if changes:
            print(f"✅ Saved {len(changes)} changes:")
            for change in changes:
                print(f"  Row {change['row']}: {change['old']} → {change['new']}")
        else:
            print("ℹ️ No changes to save (all values match original)")

def on_reset_clicked(b):
    """Handle reset button click."""
    global scenario_data
    scenario_data = copy.deepcopy(original_scenario_data)
    
    # Reset all dropdowns to original values
    for i, dropdown in enumerate(row_widgets):
        original_value = original_scenario_data[i]['result']
        if original_value is None:
            original_value = False
        dropdown.value = original_value if original_value in [True, False] else False
    
    with summary_output:
        clear_output(wait=True)
        print("🔄 Reset all values to original")

save_button.on_click(on_save_clicked)
reset_button.on_click(on_reset_clicked)

# Display the interface
print("📋 Ground Truth Review Interface")
print(f"Total scenarios: {len(scenario_data)}")
print(f"Displaying columns: {', '.join(columns_to_display)}")
print("\nUse the filter dropdown to show specific result values, then use the row dropdowns to change values.")
print("Click 'Save Changes' to update the scenario data, or 'Reset to Original' to revert all changes.\n")

display(widgets.HBox([filter_dropdown, save_button, reset_button]))
display(output_area)
display(summary_output)

# Initial display
with output_area:
    filtered = update_display('all')
    display(filtered)


## Step 5: Create Scenario Set

Create a scenario set from the reviewed scenario data. This scenario set can be used for evaluating check prompts.


In [ ]:
# Create scenario set from reviewed scenario data
from datetime import datetime
from okareo_api_client.models.scenario_set_create import ScenarioSetCreate
from okareo_api_client.models.seed_data import SeedData

# Get test run name (optional - you can also set this manually)
# If you want to fetch it from the API, uncomment the following:
# from okareo_api_client.models.general_find_payload import GeneralFindPayload
# test_runs = okareo.find_test_runs(GeneralFindPayload(id=test_run_id))
# test_run_name = test_runs[0].name if test_runs else f"test_run_{test_run_id[:8]}"
# Or set it manually:
test_run_name = f"test_run_{test_run_id[:8]}"  # Fallback: use first 8 chars of test_run_id

# Generate scenario set name: <test_run_name>_<check_name>_<timestamp>
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
scenario_set_name = f"{test_run_name}_{check_name}_{timestamp}"

# Convert scenario_data to seed_data format for SDK
# scenario_data has format: [{"input": {...}, "result": bool}, ...]
# seed_data needs: [{"input": {...}, "result": bool}, ...]
seed_data_list = []
for row in scenario_data:
    seed_data_list.append({
        "input": row["input"],  # The full nested input object with all template variables
        "result": row["result"]  # The boolean ground truth (check result)
    })

# Create SeedData objects using SDK helper method
seed_data = okareo.seed_data_from_list(seed_data_list)

# Create scenario set
create_request = ScenarioSetCreate(
    name=scenario_set_name,
    seed_data=seed_data,
    project_id=project_id
)

# Create the scenario set
print(f"Creating scenario set: {scenario_set_name}")
print(f"Number of rows: {len(seed_data)}")
print()

scenario_set_response = okareo.create_scenario_set(create_request)

print(f"✅ Scenario set created successfully!")
print(f"Scenario Set ID: {scenario_set_response.scenario_id}")
print(f"Scenario Set Name: {scenario_set_response.name}")
print(f"Scenario Set Link: {scenario_set_response.app_link}")

# Store scenario_set_id for use in subsequent phases
scenario_set_id = str(scenario_set_response.scenario_id)


## Step 6: Register Check-as-Target Model

Register the check prompt as a model/target for evaluation. This allows us to evaluate different prompt versions against the scenario set.


In [ ]:
# Get check by name - first find the check ID
all_checks = okareo.get_all_checks()
check_id = None
for c in all_checks:
    if c.name == check_name:
        check_id = str(c.id)
        break

if not check_id:
    raise ValueError(f"Check '{check_name}' not found. Available checks: {[c.name for c in all_checks]}")

# Get detailed check info
check = okareo.get_check(check_id)

print(f"✅ Found check: {check.name}")
print(f"Check ID: {check.id}")
print(f"Description: {check.description}")

# Extract prompt template from check_config
if not check.check_config:
    raise ValueError(f"Check '{check_name}' does not have a check_config")

# Convert check_config to dict if it's a Pydantic model
if hasattr(check.check_config, 'to_dict'):
    check_config = check.check_config.to_dict()
elif hasattr(check.check_config, 'dict'):
    check_config = check.check_config.dict()
elif isinstance(check.check_config, dict):
    check_config = check.check_config
else:
    # Try accessing as attributes
    check_config = {
        'prompt_template': getattr(check.check_config, 'prompt_template', None),
        'type': getattr(check.check_config, 'type', None)
    }

if "prompt_template" not in check_config or check_config.get("prompt_template") is None:
    raise ValueError(f"Check '{check_name}' does not have a prompt_template in check_config")

original_prompt_template = check_config["prompt_template"]

# Transform template variables to use "input." prefix for model registration
# Template variables that should be under input: generation, input, result, tool_calls, tools
import re

def transform_template_variables(prompt: str) -> str:
    """Transform template variables to use input. prefix for model registration."""
    # List of template variables that should be under input (excluding 'input' itself to avoid {input.input})
    template_vars = ['generation', 'result', 'tool_calls', 'tools']
    
    transformed = prompt
    for var in template_vars:
        # Match {var} but not {input.var} or {something.var}
        # Use word boundary or start of string before {, and ensure no dot before }
        pattern = r'(?<!\{)(\{)' + re.escape(var) + r'(\})(?!\.)'
        replacement = r'\1input.' + var + r'\2'
        transformed = re.sub(pattern, replacement, transformed)
    
    # Handle {input} separately - transform to {input.input}
    transformed = re.sub(r'(?<!\{)(\{input\})(?!\.)', r'{input.input}', transformed)
    
    return transformed

# Transform the prompt template to use input. prefix
transformed_prompt_template = transform_template_variables(original_prompt_template)

print(f"\n📝 Current prompt template ({len(original_prompt_template)} characters):")
print("=" * 80)
print(original_prompt_template[:500] + "..." if len(original_prompt_template) > 500 else original_prompt_template)
print("=" * 80)

if transformed_prompt_template != original_prompt_template:
    print(f"\n🔄 Transformed template variables to use 'input.' prefix:")
    print("=" * 80)
    print(transformed_prompt_template[:500] + "..." if len(transformed_prompt_template) > 500 else transformed_prompt_template)
    print("=" * 80)

# Use the transformed prompt for model registration
original_prompt_template = transformed_prompt_template


In [ ]:
# Interactive prompt editing interface
import ipywidgets as widgets
from IPython.display import display
import copy

# Store original transformed prompt for reset (already has input. prefix)
original_prompt = copy.deepcopy(original_prompt_template)

# Create text area for prompt editing
prompt_textarea = widgets.Textarea(
    value=original_prompt_template,
    placeholder='Enter your prompt template here...',
    description='Prompt:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='400px'),
    style={'description_width': 'initial'}
)

# Create buttons
save_prompt_button = widgets.Button(
    description='Save Prompt',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='150px', margin='10px 0px')
)

reset_prompt_button = widgets.Button(
    description='Reset to Original',
    button_style='warning',
    icon='undo',
    layout=widgets.Layout(width='150px', margin='10px 0px')
)

# Output area for messages
prompt_output = widgets.Output()

# Store the edited prompt (initialized to original)
edited_prompt_template = original_prompt_template

def on_save_prompt_clicked(b):
    """Handle save prompt button click."""
    global edited_prompt_template
    edited_prompt_template = prompt_textarea.value
    with prompt_output:
        prompt_output.clear_output(wait=True)
        print(f"✅ Prompt saved ({len(edited_prompt_template)} characters)")
        print(f"Changes: {'Modified' if edited_prompt_template != original_prompt else 'No changes'}")

def on_reset_prompt_clicked(b):
    """Handle reset prompt button click."""
    global edited_prompt_template
    prompt_textarea.value = original_prompt
    edited_prompt_template = original_prompt
    with prompt_output:
        prompt_output.clear_output(wait=True)
        print("🔄 Prompt reset to original")

save_prompt_button.on_click(on_save_prompt_clicked)
reset_prompt_button.on_click(on_reset_prompt_clicked)

# Display the interface
print("📝 Edit the prompt template below, then click 'Save Prompt' before registering the model.")
print("Click 'Reset to Original' to revert any changes.\n")

display(widgets.HBox([save_prompt_button, reset_prompt_button]))
display(prompt_textarea)
display(prompt_output)


In [ ]:
# Register model/target with the edited prompt
# Use the edited prompt (or original if not saved)
prompt_to_use = edited_prompt_template

# Create structured output schema matching check response format
response_format = {
    "type": "json_schema",
    "json_schema": {
        "schema": {
            "type": "object",
            "properties": {
                "score": {
                    "anyOf": [
                        {"type": "number"},
                        {"type": "boolean"}
                    ]
                },
                "explanation": {
                    "anyOf": [{"type": "string"}],
                    "description": "Explanation of the score without direct mention of the score's actual value."
                }
            },
            "required": ["score"]
        },
        "name": "CheckResponse"
    }
}

# Create dialog template with the prompt
dialog_template_content = [
    {
        "role": "system",
        "content": prompt_to_use
    }
]

# Model name: <check_name>_check_target
model_name = f"{check_name}_check_target"

# Register the model using direct API call (SDK doesn't support response_format)
# Using standard library urllib for API call
import urllib.request
import urllib.error

# Build the payload
payload = {
    "name": model_name,
    "project_id": project_id,
    "update": True,
    "models": {
        "generation": {
            "type": "generation",
            "model_id": "gpt-4.1-mini-2025-04-14",
            "temperature": 0,
            "dialog_template": json.dumps(dialog_template_content),
            "system_prompt_template": None,
            "user_prompt_template": None,
            "tools": None,
            "response_format": response_format
        }
    }
}

# Make API request
url = f"{BASE_PATH}/v0/register_model"
data = json.dumps(payload).encode('utf-8')
req = urllib.request.Request(
    url,
    data=data,
    headers={
        "Content-Type": "application/json",
        "Accept": "application/json",
        "api-key": OKAREO_API_KEY
    },
    method="POST"
)

print(f"Registering model via API: {model_name}")
print(f"Using prompt ({len(prompt_to_use)} characters)")
print()

try:
    with urllib.request.urlopen(req) as response:
        response_data = json.loads(response.read().decode('utf-8'))
        model_id = response_data.get("id")
        
        if model_id:
            print(f"✅ Model registered successfully!")
            print(f"Model ID: {model_id}")
            print(f"Model Name: {response_data.get('name', model_name)}")
            print(f"Model Version: {response_data.get('version', 'N/A')}")
            
            # Store model_id for use in subsequent phases
            model_id = str(model_id)
        else:
            raise ValueError(f"Could not extract model ID from response: {response_data}")
except urllib.error.HTTPError as e:
    error_body = e.read().decode('utf-8')
    raise Exception(f"API request failed with status {e.code}: {error_body}")
except Exception as e:
    raise Exception(f"Error registering model: {str(e)}")


## Step 7: Create Score Extraction Check (check_value_match)

Create a custom check that extracts the score from the check-as-target model's JSON output. The check-as-target model returns JSON in the format `{"score": bool/float, "explanation": "..."}`, and this check extracts just the `score` field for evaluation.


In [ ]:
# Create check_value_match check to extract score from JSON output
import json
import urllib.request
import urllib.error

# Manually construct the code string for the check
check_code = '''import json

from okareo.checks import CodeBasedCheck


def get_pass_fail(text):
    """
    Replicate the get_pass_fail() logic from nlg_utils.py
    Converts various input types to boolean following the server's logic.
    """
    if isinstance(text, bool):
        return text
    if isinstance(text, (int, float)):
        return bool(text)  # Convert number to bool (0/0.0 -> False, any non-zero -> True)
    if isinstance(text, str):
        if "true" in text or "True" in text:
            return True
        if "false" in text or "False" in text:
            return False
        if "0" in text:
            return False
        if "1" in text:
            return True
    return text


class Check(CodeBasedCheck):

    @staticmethod
    def evaluate(model_output: str | dict, scenario_input: str, scenario_result: str, metadata: dict) -> bool:
        """
        Evaluates if the score extracted from structured JSON model output matches the scenario result.
        
        Steps:
        1. Parse model_output as JSON (if string) or use directly (if dict)
        2. Extract 'score' field from the JSON
        3. Apply get_pass_fail() conversion to get boolean
        4. Convert boolean to lowercase string ("true"/"false")
        5. Compare to scenario_result string ("true" or "false")
        
        Returns True if they match, False otherwise.
        """
        try:
            # Parse model_output as JSON if it's a string
            if isinstance(model_output, str):
                try:
                    parsed_output = json.loads(model_output)
                except json.JSONDecodeError:
                    # If JSON parsing fails, return False
                    return False
            elif isinstance(model_output, dict):
                parsed_output = model_output
            else:
                # Unexpected type, return False
                return False
            
            # Extract score field
            score = parsed_output.get("score")
            if score is None:
                # Score field missing, return False
                return False
            
            # Apply get_pass_fail() conversion to get boolean
            score_bool = get_pass_fail(score)
            
            # Ensure we have a boolean (get_pass_fail might return the original value in some cases)
            if not isinstance(score_bool, bool):
                # If conversion didn't produce a boolean, return False
                return False
            
            # Convert boolean to lowercase string
            score_str = str(score_bool).lower()  # "True" -> "true", "False" -> "false"
            
            # Parse scenario_result as string (it should be "true" or "false")
            scenario_result_str = str(scenario_result).strip().lower()
            
            # Compare
            return score_str == scenario_result_str
            
        except Exception:
            # Any unexpected error, return False
            return False
'''

score_extraction_check_name = "check_value_match"
check_description = "Extracts the score field from JSON output of check-as-target models. Used in meta-evaluation workflows."

# Construct the payload
payload = {
    "name": score_extraction_check_name,
    "description": check_description,
    "check_config": {
        "code_contents": check_code,
        "type": "pass_fail"  # Returns a boolean (pass/fail)
    },
    "project_id": project_id
}

url = f"{BASE_PATH}/v0/check_create_or_update"
data = json.dumps(payload).encode('utf-8')
req = urllib.request.Request(
    url,
    data=data,
    headers={
        "Content-Type": "application/json",
        "Accept": "application/json",
        "api-key": OKAREO_API_KEY
    },
    method="POST"
)

print(f"Creating check: {score_extraction_check_name}")
print(f"Description: {check_description}")
print()

try:
    with urllib.request.urlopen(req) as response:
        response_data = json.loads(response.read().decode('utf-8'))
        check_id = response_data.get("id")
        
        if check_id:
            print(f"✅ Check created/updated successfully!")
            print(f"Check ID: {check_id}")
            print(f"Check Name: {response_data.get('name', score_extraction_check_name)}")
        else:
            raise ValueError(f"Could not extract check ID from response: {response_data}")
except urllib.error.HTTPError as e:
    error_body = e.read().decode('utf-8')
    raise Exception(f"API request failed with status {e.code}: {error_body}")
except Exception as e:
    raise Exception(f"Error creating check: {str(e)}")

## Step 8: Run Evaluation

Submit the evaluation using the scenario set and registered model. This will run the check-as-target model against all scenarios in the scenario set.


In [ ]:
# Run evaluation: Execute check-as-target model against scenario set using SDK
from datetime import datetime
from okareo.model_under_test import TestRunType

# Generate evaluation name
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
evaluation_name = f"{check_name}_eval_{timestamp}"

# Get the ModelUnderTest object using SDK's get_model method
model_name = f"{check_name}_check_target"
model = okareo.get_model(model_name, version="latest")

# Run the evaluation using SDK's run_test method (synchronous, waits for completion)
print(f"Running evaluation: {evaluation_name}")
print(f"Model Name: {model_name}")
print(f"Model ID: {model_id}")
print(f"Scenario Set ID: {scenario_set_id}")
print("This may take a few minutes...")
print()

test_run = model.run_test(
    scenario=scenario_set_id,  # Can pass scenario ID as string
    name=evaluation_name,
    api_keys={"generation": OPENAI_API_KEY},
    test_run_type=TestRunType.NL_GENERATION,
    calculate_metrics=True,
    checks=[score_extraction_check_name]  # Use check_value_match to extract scores from JSON outputs
)

print(f"✅ Evaluation completed successfully!")
print(f"Test Run ID: {test_run.id}")
print(f"Status: {test_run.status if hasattr(test_run, 'status') else 'completed'}")
if hasattr(test_run, 'end_time'):
    print(f"End time: {test_run.end_time}")
if hasattr(test_run, 'app_link'):
    print(f"Link: {test_run.app_link}")

# Store test_run_id and test_run object for use in subsequent phases
evaluation_test_run_id = str(test_run.id)
final_test_run = test_run
